# LG M50 Single Pulse Parameter Identification

## Validating the ECM parameters identified from a single pulse

This example presents an experimental parameter identification method for a two-RC circuit model. The data for this notebook is located within the same directory and was obtained from WDWidanage/Simscape-Battery-Library [[1]](https://github.com/WDWidanage/Simscape-Battery-Library/tree/a3842b91b3ccda006bc9be5d59c8bcbd167ceef7/Examples/parameterEstimation_TECMD/Data).

### Setting up the Environment

If you don't already have PyBOP installed, check out the [installation guide](https://pybop-docs.readthedocs.io/en/latest/installation.html) first.

We begin by importing the necessary libraries. Let's also fix the random seed to generate consistent output during development.

In [ ]:
import numpy as np
import polars as pl
import pybamm
from scipy.io import loadmat

import pybop

np.random.seed(8)  # users can remove this line

## Importing data

The data is imported as a dictionary with the following key level:
- ["LGM50_5Ah_Pulse"]
    - Temperature-> ["T0"]
        - State of Charge ->["SoC3"]
            - Cell number -> ["Cell1"]
                - Corresponding data -> ["data"]

Using SciPy's `loadmat`, we import the data from the MAT containers:

In [ ]:
ocp = loadmat(
    "../../data/LG_M50_ECM/data/LGM50_5Ah_OCV.mat", simplify_cells=True, mat_dtype=False
)
pulse_data = loadmat(
    "../../data/LG_M50_ECM/data/LGM50_5Ah_Pulse.mat",
    simplify_cells=True,
    mat_dtype=False,
)
rate_data = loadmat(
    "../../data/LG_M50_ECM/data/LGM50_5Ah_RateTest.mat",
    simplify_cells=True,
    mat_dtype=False,
)

### Filter data using polars

Next, we construct a dataframe from a selected pulse. In this case, we select the data for zero degrees (`T0`) with a state-of-charge of 90% (`SoC9`) and the 19th cell (`Cell19`). This can be extended to multiple pulses, or a different selection.

Additionally, we apply two filters to the dataframe to ensure the data contains only monotonically increasing time samples without duplicates.

We can then form the `pybop.Dataset` from the experimental data. In this situation, the applied current convention is different between the experimental data and PyBOP's convention. This is solved by negating the vector on construction.

In [ ]:
lf = pl.LazyFrame(pulse_data["LGM50_5Ah_Pulse"]["T0"]["SoC9"]["Cell19"]["data"])
column_renaming = {
    "ProgTime": "Time [s]",
    "Voltage": "Voltage [V]",
    "Current": "Negative current [A]",
}
lf = (
    lf.rename(column_renaming)
    .unique(subset=["Time [s]"], keep="first", maintain_order=True)
    .with_columns((pl.col("Negative current [A]") * -1).alias("Current [A]"))
)
dataset = pybop.Dataset(lf)

A plot of voltage vs. time confirms the data looks as expected for fitting. In this situation, we would prefer to have additional samples from the relaxation, but as we will show below, PyBOP is still able to identify parameter values that fit this system.

In [ ]:
pybop.plot.dataset(dataset);

Next, we construct the OCV function from the imported OCV data. This is completed with a wrapper method on the `pybamm.Interpolant` function,

In [ ]:
ocv_LGM50 = pybop.Interpolant(
    ocp["LGM50_5Ah_OCV"]["T25"]["refSoC"].squeeze() / 100,
    ocp["LGM50_5Ah_OCV"]["T25"]["meanOCV"].squeeze(),
    name="OCV",
)

We can construct the model and parameter values for a two-RC circuit model with initial values as listed below. Note, the "Initial SoC" is shifted slightly to better match the zero degree data.

In [ ]:
model = pybamm.equivalent_circuit.Thevenin(options={"number of rc elements": 2})
parameter_values = pybamm.ParameterValues(
    {
        "chemistry": "ecm",
        "Initial SoC": 0.9 - 0.01,
        "Initial temperature [K]": 25 + 273.15,
        "Cell capacity [A.h]": 5,
        "Nominal cell capacity [A.h]": 5,
        "Ambient temperature [K]": 25 + 273.15,
        "Current function [A]": 4.85,
        "Upper voltage cut-off [V]": 4.2,
        "Lower voltage cut-off [V]": 3.0,
        "Cell thermal mass [J/K]": 1000,
        "Cell-jig heat transfer coefficient [W/K]": 10,
        "Jig thermal mass [J/K]": 500,
        "Jig-air heat transfer coefficient [W/K]": 10,
        "Open-circuit voltage [V]": ocv_LGM50,
        "R0 [Ohm]": 0.005,
        "Element-1 initial overpotential [V]": 0,
        "Element-2 initial overpotential [V]": 0,
        "R1 [Ohm]": 0.0001,
        "R2 [Ohm]": 0.0001,
        "C1 [F]": 3000,
        "C2 [F]": 6924,
        "Entropic change [V/K]": 0.0004,
    }
)

## Identifying the parameters

Now that the initial parameter values and model are defined, we can start the PyBOP fitting process. In this example, we are going to try to fit all five resistance and capacitance parameters at once. To do this, we define a `pybop.Parameter` for each fitting parameter and compile them into a list.

In [ ]:
parameter_values.update(
    {
        "R0 [Ohm]": pybop.Parameter(
            pybop.Gaussian(0.005, 0.0001, truncated_at=[1e-6, 2e-1]),
        ),
        "R1 [Ohm]": pybop.Parameter(
            pybop.Gaussian(0.0001, 0.0001, truncated_at=[1e-6, 1]),
        ),
        "R2 [Ohm]": pybop.Parameter(
            pybop.Gaussian(0.0001, 0.0001, truncated_at=[1e-6, 1]),
        ),
        "C1 [F]": pybop.Parameter(
            pybop.Gaussian(3000, 2500, truncated_at=[0.5, 1e4]),
        ),
        "C2 [F]": pybop.Parameter(
            pybop.Gaussian(3000, 2500, truncated_at=[0.5, 1e4]),
        ),
    }
)

The `Problem` class provides us with a class to hold all of the objects we need to evaluate our selected `SumSquaredError` cost function. 

In [ ]:
simulator = pybop.pybamm.Simulator(
    model, parameter_values=parameter_values, protocol=dataset, cache_esoh=False
)
cost = pybop.SumSquaredError(dataset)
problem = pybop.Problem(simulator, cost)

The cost function can be interrogated via the `problem.evaluate` function as follows.

In [ ]:
inputs = problem.parameters.to_dict([0.01, 0.01, 0.01, 10000, 10000])
evaluation = problem.evaluate(inputs)
print(evaluation.get_values()[0])

1.1491419699687517


Next, we construct the optimisation class with our algorithm of choice and run it. In this case, we select the PSO method as it provides global optimisation capability. After optimisation, we can compare the initial parameters to the identified parameters.

In [ ]:
options = pybop.PintsOptions(max_unchanged_iterations=55, threshold=1e-6)
optim = pybop.PSO(problem, options=options)
result = optim.run()

## Plotting and visualisation

PyBOP provides various plotting utilities to visualise the results of the optimisation. This time, let's look at the fitting results and validation against a second dataset.

Here we compare the optimised model prediction against the original dataset.

In [ ]:
pybop.plot.problem(problem, inputs=result.best_inputs, title="Optimised Comparison");

### Convergence and parameter trajectories

To assess the optimisation process, we can plot the convergence of the cost function and the trajectories of the parameters:

In [ ]:
result.plot_convergence()
result.plot_parameters();

## Validating the fit

We've identified parameters from a single discharge pulse. Now, let's ask ourselves how well these parameters extrapolate to other operational conditions. To accomplish this, let's load a new pulse dataset at 80% SOC and compare it to our identified forward model.

In [ ]:
lf_pulse_two = pl.LazyFrame(
    pulse_data["LGM50_5Ah_Pulse"]["T0"]["SoC8"]["Cell19"]["data"]
)
lf_pulse_two = (
    lf_pulse_two.rename(column_renaming)
    .unique(subset=["Time [s]"], keep="first", maintain_order=True)
    .with_columns((pl.col("Negative current [A]") * -1).alias("Current [A]"))
)

dataset_two_pulse = pybop.Dataset(lf_pulse_two)

Now that we have a new dataset, we update the target within the problem class as well as the "Initial SoC" value. Once that has been completed, we rebuild the problem.

In [ ]:
simulator = pybop.pybamm.Simulator(
    model,
    parameter_values=parameter_values,
    protocol=dataset_two_pulse,
    initial_state={"Initial SoC": 0.8 - 0.0075},
    cache_esoh=False,
)
cost_two_pulse = pybop.SumSquaredError(dataset_two_pulse)
problem = pybop.Problem(simulator, cost_two_pulse)

Let's plot the simulation against the new pulse data:

In [ ]:
pybop.plot.problem(problem, inputs=result.best_inputs, title="Parameter Extrapolation");

## Concluding thoughts

As expected, when identifying parameters from a single pulse, extrapolation to different operating conditions is challenging. To address this issue, parameter identification with various pulse datasets is recommended.

This notebook illustrates how to perform circuit model parameter identification using PSO in PyBOP, providing insights into the optimisation process through various visualisations.